# 01 — Data Importing and Overview
### Introduction:

This notebook imports the raw exploration datasets from the 2023 MPD drill program and performs an initial inspection before any cleaning or transformation begins.


### Objectives:
- Load raw drillhole, assay, and collar datasets
- Standardize column names across tables
- Verify dataset integrity and hole ID alignment
- Export minimally standardized tables for Notebook 02

### Contents:
**1. Setup and Libraries**

**2. Load Raw CSV Files**

**3. Add Collar Table**

**4. Inspect Raw Table Structure**

**5. Standardize Column Names**

**6. Align Key Drill Log Columns**

**7. Preview Cleaned Raw Tables**

**8. Verify Dataset Integrity**

**9. Compare Hole IDs Across Datasets**

**10. Export Standardized Tables**

**11. Conclusions**


### 1. Setup and libraries:

The first step is to import the libraries we need and define the raw data path. Because this notebook lives in the `Notebooks` folder, the `Data/raw` folder is one level up.


In [ ]:
import warnings
from pathlib import Path
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

# The notebook folder is /Notebooks, so the project root is the parent folder
project_root = Path().resolve().parent
raw_data = project_root / "Data" / "raw"

raw_data


### 2. Load raw CSV files:

This project uses several raw tables: drill logs, drill assays, soil samples, soil assays, rock descriptions, rock assays, and manually transcribed collar data. We start by loading the CSV files directly from the raw data folder.


In [ ]:
def load_csv(filename, header_row=0):
    df = pd.read_csv(raw_data / filename, header=header_row)
    df = df.loc[:, ~df.columns.str.contains(r"^Unnamed")]
    return df

drill_df = load_csv("Appendix A_2023 Drill Data.csv", header_row=1)
assay_df = load_csv("Appendix B_2023 DDH Assays.csv", header_row=0)
soil_data = load_csv("Appendix E_2023 Soil Data.csv", header_row=0)
soil_assays = load_csv("Appendix F_2023 Soil Assays.csv", header_row=0)
rock_data = load_csv("Appendix H_2023 Rock Data.csv", header_row=0)
rock_assays = load_csv("Appendix I_2023 Rock Assays.csv", header_row=0)


In [ ]:
raw_tables = {
    "drill_df": drill_df,
    "assay_df": assay_df,
    "soil_data": soil_data,
    "soil_assays": soil_assays,
    "rock_data": rock_data,
    "rock_assays": rock_assays,
}

for name, df in raw_tables.items():
    print(f"{name}: {df.shape[0]} rows x {df.shape[1]} columns")


### 3. Add collar table:

Collar coordinates are not available in the same CSV files, so this table was manually transcribed from the ARIS assessment PDF report. Collar data is required for later spatial validation and integration.


In [ ]:
collar_data = [
    ["AXE-23-001", 677400, 5503115, 1418, 180, -65, 732, "West", "Completed"],
    ["AXE-23-002", 677400, 5503115, 1418,   0, -90, 819.01, "West", "Completed"],
    ["AXE-23-003", 677400, 5503115, 1418,  90, -45, 367, "West", "Completed"],
    ["AXE-23-004", 677400, 5503115, 1418,  90, -75, 707, "West", "Completed"],
    ["AXE-23-005", 677400, 5503115, 1418,  25, -50, 87.35, "West", "Abandoned"],
    ["AXE-23-006", 677400, 5503120, 1418,  15, -45, 97.35, "West", "Abandoned"],
    ["AXE-23-007", 677400, 5503120, 1418,  15, -50, 459, "West", "Completed"],
    ["AXE-23-008", 677397, 5502825, 1398, 350, -75, 897, "West", "Completed"],
    ["AXE-23-009", 677397, 5502825, 1398,  90, -85, 83, "West", "Abandoned"],
    ["AXE-23-010", 677397, 5502825, 1398,  90, -80, 709.1, "West", "Completed"],
    ["AXE-23-011", 677397, 5502930, 1398,   0, -90, 1031, "West", "Completed"],
    ["AXE-23-012", 678515, 5501650, 1310, 100, -68, 849, "South", "Completed"],
    ["AXE-23-013", 678515, 5501650, 1335, 305, -80, 944, "South", "Completed"],
    ["AXE-23-014", 678515, 5501650, 1310, 345, -57, 1061.2, "South", "Completed"],
    ["AXE-23-015", 680135, 5501665, 1000,   0, -90, 253, "1516", "Ended"],
    ["AXE-23-016", 680135, 5501665, 1000,  90, -80, 782, "1516", "Completed"],
    ["AXE-23-017", 680135, 5501665, 1000,  90, -50, 600, "1516", "Completed"],
    ["AXE-23-018", 680135, 5501665, 1000, 20.17, -51, 938, "1516", "Completed"],
    ["AXE-23-019", 680139, 5501677,  995, 145.35, -50, 146, "1516", "Ended"],
    ["AXE-23-020", 680139, 5501677,  995, 145, -60, 129, "1516", "Ended"],
    ["AXE-23-021", 680344, 5501891, 1150, 290, -75, 183, "1516", "Ended"],
    ["MPD-23-001", 681435, 5513816, 1360,   0, -90, 995, "Man", "Completed"],
    ["MPD-23-002", 681435, 5513816, 1360,  90, -70, 924, "Man", "Completed"],
    ["MPD-23-003", 681435, 5513816, 1360,  90, -80, 1094, "Man", "Completed"],
    ["MPD-23-004", 681434, 5513814, 1360, 272, -60, 104, "Man*", "Abandoned"],
    ["MPD-23-005", 681434, 5513814, 1360, 272, -65, 825, "Man*", "Completed"],
    ["MPD-23-006", 681435, 5513816, 1360, 272, -80, 879, "Man", "Completed"],
    ["MPD-23-007", 681435, 5513816, 1360, 342.6, -50, 807, "Man", "Completed"],
    ["MPD-23-008", 681360, 5513440, 1364,  90, -45, 54, "Beyer", "Abandoned"],
    ["MPD-23-009", 681360, 5513440, 1364,  90, -60, 487, "Beyer", "Completed"],
    ["MPD-23-010", 681416, 5513389, 1387, 343, -65, 295, "Beyer", "Completed"],
    ["MPD-23-011", 681416, 5513389, 1387, 333, -68, 80, "Beyer", "Completed"],
    ["MPD-23-012", 681416, 5513389, 1387, 333, -50, 144, "Beyer", "Completed"],
]

collar_df = pd.DataFrame(collar_data, columns=[
    "hole_id", "easting", "northing", "elevation_m",
    "azimuth_deg", "dip_deg", "length_m", "area", "status"
])

collar_df.head()


### 4. Inspect raw table structure:

We want to confirm the initial shape and column names before performing any structural standardization.


In [ ]:
print("drill_df shape:", drill_df.shape)
print("assay_df shape:", assay_df.shape)
print("collar_df shape:", collar_df.shape)

print("Drill columns:")
print(drill_df.columns.tolist())
print("Assay columns:")
print(assay_df.columns.tolist())


### 5. Standardize column names:

The drill and assay tables use different column formatting conventions. We standardize names now to make later comparisons easier.


In [ ]:
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.replace("
", " ", regex=False)
        .str.strip()
        .str.replace(r"\s+", "_", regex=True)
        .str.replace(r"[^0-9a-zA-Z_]", "", regex=True)
        .str.lower()
    )
    return df

drill_df = clean_columns(drill_df)
assay_df = clean_columns(assay_df)

print("Standardized drill columns:")
print(drill_df.columns.tolist())
print("
Standardized assay columns:")
print(assay_df.columns.tolist())

### 6. Align key drill log columns:

The drill log uses `from_m` and `to_m`, while the assay table uses `from` and `to`. We rename the drill log columns so the two tables align more naturally for later merging.


In [ ]:
drill_df = drill_df.rename(columns={"from_m": "from", "to_m": "to"})
print("drill_df columns after renaming:")
print(drill_df.columns.tolist())


### 7. Preview the cleaned raw tables:

A quick preview helps confirm the import and standardization steps.


In [ ]:
display(drill_df.head(5))
display(assay_df.head(5))


### 8. Verify dataset integrity:

Now we check for completely empty rows and missing hole identifiers in the drill and assay tables.


In [ ]:
print("Drill records with all values missing:", drill_df.isna().all(axis=1).sum())
print("Assay records with all values missing:", assay_df.isna().all(axis=1).sum())

missing_drill_holes = drill_df[drill_df["hole_number"].isna()]
missing_assay_holes = assay_df[assay_df["hole_number"].isna()]

print("Drill rows with missing hole_number:", missing_drill_holes.shape[0])
print("Assay rows with missing hole_number:", missing_assay_holes.shape[0])


### 9. Compare hole IDs across drill and assay datasets:

We want to understand whether any holes exist only in the drill log or only in the assay table.


In [ ]:
drill_holes = set(drill_df["hole_number"].dropna().unique())
assay_holes = set(assay_df["hole_number"].dropna().unique())

print("Holes in drill log but not in assay table:", sorted(drill_holes - assay_holes))
print("Holes in assay table but not in drill log:", sorted(assay_holes - drill_holes))


### 10. Export the minimally standardized raw tables:

These files preserve the original raw datasets with only basic structural standardization. Notebook 02 will perform the deeper cleaning and table splitting.


In [ ]:
drill_df.to_csv(raw_data / "drill_raw.csv", index=False)
assay_df.to_csv(raw_data / "assay_raw.csv", index=False)
collar_df.to_csv(raw_data / "collar_raw.csv", index=False)
print("Saved drill_raw.csv, assay_raw.csv, collar_raw.csv")


### 11. Conclusions:

- Seven raw tables were successfully imported from the 2023 MPD ARIS assessment report appendices — drill logs, assays, soil samples, soil assays, rock descriptions, rock assays, and collar coordinates
- Column names were standardized across the drill and assay tables using a consistent lowercase underscore convention, making later merging straightforward
- Collar coordinates were manually transcribed from the ARIS PDF and added as a structured table — covering all 26 drillholes with easting, northing, elevation, azimuth, dip, and total depth
- Drill rows with missing hole identifiers were identified — these result from Excel merged cells in the original source file and are resolved in Notebook 02 using forward fill
- All holes present in the drill log were confirmed in the assay table and vice versa — no holes are missing from either dataset
- Minimally standardized tables are exported to Data/raw and are ready for the deeper geology-aware cleaning in Notebook 02